# Qwen 后训练 (QLoRA + SFT)

一键在 Colab 或 Kaggle 上微调千问模型。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/greathousesh/post-training/blob/main/notebooks/train_qwen.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/post-training/blob/main/notebooks/train_qwen.ipynb)

**使用前**：把这个 notebook 里所有的 `greathousesh/post-training` 改成你自己的仓库地址（包括上面两个 badge 链接 和下方 git clone 那一格）。

**Kaggle 注意**：右侧 Settings → Accelerator 选 GPU (T4 x2)，Internet 打开（下载模型需要）。

**Colab 注意**：菜单栏 Runtime → Change runtime type → Hardware accelerator 选 GPU。

## 1. 环境检查

In [ ]:
!nvidia-smi

## 2. 安装依赖

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets pyyaml

## 3. 拉取项目代码

把 `REPO_URL` 改成你的 GitHub 仓库。

In [ ]:
import os

REPO_URL = "https://github.com/greathousesh/post-training.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
!ls

## 4. 准备数据

数据格式：JSONL，每行一条对话，结构见 `data/example.jsonl`。

- **Colab**：下一格可以上传自己的 `train.jsonl`
- **Kaggle**：用右侧 `+ Add Input` 挂载一个 Dataset，然后把路径改成 `/kaggle/input/<dataset>/train.jsonl`
- **想先跑通流程**：跳过本步，直接用 `data/example.jsonl`

In [ ]:
# Colab 专用：上传自定义数据（取消注释使用）
# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, f"data/{name}")

!head -1 data/example.jsonl

## 5. 按 GPU 自适应调整 config

根据当前 GPU 显存自动选择模型大小和 batch 配置，改写 `config.yaml`。

In [ ]:
import yaml, torch

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {vram_gb:.1f} GB")

# 你也可以改下面这行指定训练数据
cfg["data"]["train_file"] = "data/example.jsonl"

if vram_gb < 20:            # T4 16GB / P100 16GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-1.5B-Instruct"
    cfg["model"]["max_length"] = 1024
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 16
elif vram_gb < 30:          # L4 22GB / V100 32GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-7B-Instruct"
    cfg["model"]["max_length"] = 1024
    cfg["training"]["per_device_train_batch_size"] = 1
    cfg["training"]["gradient_accumulation_steps"] = 16
else:                       # A100 40/80GB
    cfg["model"]["name"] = "Qwen/Qwen2.5-7B-Instruct"

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

print(yaml.dump(cfg, allow_unicode=True, sort_keys=False))

## 6. 训练

In [ ]:
!python train.py --config config.yaml

## 7. 导出 / 下载 LoRA adapter

In [ ]:
!ls outputs/qwen-sft-lora
!tar -czf qwen-lora.tar.gz outputs/qwen-sft-lora

try:
    from google.colab import files
    files.download("qwen-lora.tar.gz")
except ImportError:
    print("Kaggle: 文件已保存到 /kaggle/working/qwen-lora.tar.gz，右侧 Output 面板下载")

## 8. (可选) 合并 LoRA 到完整权重

合并需要以 bf16 加载完整 base model（7B 约 14GB），T4/P100 显存不够，建议在 A100 或本地做。

In [ ]:
# !python -m src.merge \
#     --base Qwen/Qwen2.5-7B-Instruct \
#     --adapter outputs/qwen-sft-lora \
#     --output outputs/qwen-sft-merged